# 03 — Feature Ablation and Operational Telemetry

## Research Question
Which telemetry channels carry the strongest detection signal?

## Hypothesis
Operational telemetry alone should provide a non-trivial risk signal, while full telemetry should improve robustness and interpretability.

## Input Data
- `episode_df_all`
- `early_df_all` when prefix-level ablations are used

## Prediction/Analysis Target
- `is_attack`, `attack_type`, and prefix-level attack labels

## Validation Protocol
Feature-group ablations under leave-one-seed-out evaluation.

## Expected Paper Artifact
- Ablation tables
- Operational-only comparison plots
- Prefix-level ablation curves


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import silhouette_score
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    make_scorer,
)

from evaluation import (
    load_jsonl, 
    evaluate_binary_prediction, 
    evaluate_multiclass_attack_type,
    fit_rf_feature_importance,
    evaluate_multiclass_feature_group_ablation,
    add_akc_phase_from_attack_type,
    normalize_seed_column,
    infer_telemetry_features,
    prepare_akc_phase_df,
    run_akc_phase_leave_one_seed,
    summarize_multiclass_results,
    collect_akc_phase_predictions_leave_one_seed,
    plot_confusion_matrix_from_predictions,
    infer_early_akc_features,
    classification_report,
    run_temporal_akc_phase_classification,
    plot_temporal_akc_curves,
    adversarial_fragmentation_index,
    risk_conditioned_fragmentation,
    semantic_fragmentation_proxy,
    add_benign_normalized_fragmentation,
    evaluate_leave_one_seed_out,
    TELEMETRY_NUMERIC,
    evaluate_feature_group_ablation,
    FEATURE_GROUPS,
    evaluate_leave_one_attack_out,
    compute_permutation_importance,
    run_operational_only_leave_one_seed,
    summarize_results,
    plot_exp1_operational_comparison,
    load_tamas_jsonl,
    build_early_df_from_agent_outputs,
    run_early_detection_leave_one_seed,
    plot_early_detection_curve,
    EARLY_FEATURES_CANDIDATES,
    validate_early_df_for_attack_analysis,
    run_early_detection_by_attack_type,
    summarize_early_detection_by_attack,
    plot_metric_curves_by_attack,
    compute_temporal_gain,
    infer_numeric_features,
    keep_existing_numeric,
    run_early_detection_feature_ablation,
    plot_ablation_curves,
    telemetry_guardrail_decision,
    run_akc_phase_leave_one_attack_type_out,
)
from datasets.tamas.tamas_features import build_all_feature_tables

RESULTS_DIR = Path("results/tamas")
RAW_DIR = RESULTS_DIR / "raw"
FEATURES_DIR = RESULTS_DIR / "features"
PLOTS_DIR = RESULTS_DIR / "plots"

scenario = "education"
architecture = "centralized_tamas"
CONDITIONS = [
    "benign",
    "byzantine",
    "colluding",
    "contradicting",
    "DPI",
    "impersonation",
    "IPI",
]
SEEDS = [1, 2, 3]
MODEL_NAMES = [
    "ticlazau/meta-llama-3.1-8b-instruct:latest",
    # "qwen3",
]

In [ ]:

# Load processed telemetry tables generated by 00_setup_and_feature_extraction.ipynb.
# If files are not available yet, run notebook 00 first.
PROCESSED_DIR = Path("results/tamas/processed")

for _name in ["episode_df_all", "agent_df_all", "early_df_all", "df_raw_all"]:
    _path = PROCESSED_DIR / f"{_name}.parquet"
    if _path.exists():
        globals()[_name] = pd.read_parquet(_path)
        print(f"Loaded {_name}: {globals()[_name].shape}")
    else:
        print(f"Missing {_path}; run 00_setup_and_feature_extraction.ipynb if this notebook needs it.")


### Experimento 1 — Operational-only telemetry

Objetivo: testar se a detecção funciona sem markers linguísticos diretos, usando apenas telemetria operacional: tokens, latência, número de mensagens, tools e dinâmica estrutural.

Este é crucial para defender o claim de “telemetry-based safety”.

In [ ]:
exp1_results = run_operational_only_leave_one_seed(episode_df_all)

exp1_summary = summarize_results(exp1_results, group_cols=["feature_set", "model"])
print(exp1_summary)

plot_exp1_operational_comparison(exp1_summary)

In [ ]:
METADATA_COLS = {
    "is_attack",
    "attack_specific_label",
    "trace_fraction",
    "seed",
    "task_id",
    "episode_id",
    "benchmark",
    "architecture",
    "model_name",
    "scenario",
    "attack_type",
    "expected_label",
    "source_file",
    "hard_benign_quantile",
    "hard_benign_threshold",
}

early_df_all = normalize_seed_column(early_df)
early_df_all["is_attack"] = early_df_all["is_attack"].astype(int)

ALL_EARLY_FEATURES = infer_numeric_features(early_df_all)

In [ ]:
STRUCTURAL_PREFIX_FEATURES = {
    "prefix_steps",
    "total_steps",
    "observed_step_ratio",
    "num_observed_steps",
}

AGENT_STRUCTURE_FEATURES = {
    "num_unique_roles_prefix",
    "num_unique_agents_prefix",
    "num_observed_steps",
    "num_llm_calls_prefix",
    "prefix_steps",
    "total_steps",
    "observed_step_ratio",
}

TOOL_STRUCTURE_FEATURES = {
    "num_requested_tools_prefix",
    "num_executed_tools_prefix",
    "tool_execution_rate_prefix",
}

TEXT_LENGTH_FEATURES = {
    "final_answer_words_prefix_sum",
    "final_answer_words_prefix_mean",
    "final_answer_words_prefix_std",
}

OPERATIONAL_COST_LATENCY_TOKEN_FEATURES = {
    "prompt_tokens_prefix_sum",
    "response_tokens_prefix_sum",
    "total_tokens_prefix_sum",
    "prompt_tokens_prefix_mean",
    "response_tokens_prefix_mean",
    "total_tokens_prefix_mean",
    "prompt_tokens_prefix_std",
    "response_tokens_prefix_std",
    "total_tokens_prefix_std",
    "latency_prefix_sum",
    "latency_prefix_mean",
    "latency_prefix_std",
    "eval_duration_prefix_sum",
    "eval_duration_prefix_mean",
    "eval_duration_prefix_std",
    "prompt_eval_duration_prefix_sum",
    "prompt_eval_duration_prefix_mean",
    "tokens_per_second_prefix_mean",
    "tokens_per_second_prefix_std",
    "tokens_per_second_prefix_min",
    "tokens_per_second_prefix_max",
    "prompt_eval_count_prefix_sum",
    "eval_count_prefix_sum",
    "token_budget_prefix_sum",
    "token_budget_prefix_mean",
    "response_to_prompt_ratio_prefix",
    "tokens_per_latency_prefix",
}

FEATURE_SETS_ABLATION = {
    "all_early_features": ALL_EARLY_FEATURES,

    "without_prefix_progress_features": [
        f for f in ALL_EARLY_FEATURES
        if f not in STRUCTURAL_PREFIX_FEATURES
    ],

    "operational_cost_latency_tokens_only": keep_existing_numeric(
        early_df_all,
        OPERATIONAL_COST_LATENCY_TOKEN_FEATURES,
    ),

    "without_agent_structure_features": [
        f for f in ALL_EARLY_FEATURES
        if f not in AGENT_STRUCTURE_FEATURES
    ],

    "without_agent_and_tool_structure": [
        f for f in ALL_EARLY_FEATURES
        if f not in AGENT_STRUCTURE_FEATURES
        and f not in TOOL_STRUCTURE_FEATURES
    ],

    "without_text_length_features": [
        f for f in ALL_EARLY_FEATURES
        if f not in TEXT_LENGTH_FEATURES
    ],
}

# for name, features in FEATURE_SETS_ABLATION.items():
#     print(f"\n{name}: {len(features)} features")
#     print(features)

In [ ]:
early_ablation_results = run_early_detection_feature_ablation(
    early_df=early_df_all,
    feature_sets=FEATURE_SETS_ABLATION,
    model_kind="rf",
)

early_ablation_summary = summarize_results(
    early_ablation_results,
    group_cols=["feature_set", "trace_fraction"],
)

# early_ablation_summary

In [ ]:
plot_ablation_curves(
    early_ablation_summary,
    metric="roc_auc_mean",
    output_path=OUTPUT_DIR / "early_detection_ablation_curves_roc_auc.png",
)

plot_ablation_curves(
    early_ablation_summary,
    metric="balanced_accuracy_mean",
    output_path=OUTPUT_DIR / "early_detection_ablation_curves_balanced_accuracy.png",
)

In [ ]:
# roc_auc_ablation_pivot = early_ablation_summary.pivot(
#     index="feature_set",
#     columns="trace_fraction",
#     values="roc_auc_mean",
# )

# bacc_ablation_pivot = early_ablation_summary.pivot(
#     index="feature_set",
#     columns="trace_fraction",
#     values="balanced_accuracy_mean",
# )

# roc_auc_ablation_pivot

### Fazer ablação específica para is_attack

Você precisa mostrar qual grupo de telemetria sustenta a detecção.

In [ ]:
# FEATURE_GROUPS = {
#     "cost": [
#         "prompt_tokens",
#         "response_tokens",
#         "total_tokens",
#         "latency_total_s",
#         "num_llm_calls",
#     ],
#     "tool_use": [
#         "num_tool_calls",
#         "unique_tool_calls",
#         "tool_call_entropy",
#         "first_tool_call_position",
#         "num_suspicious_tool_calls",
#         "used_any_suspicious_tool",
#     ],
#     "linguistic_markers": [
#         "injection_marker_count",
#         "refusal_marker_count",
#         "contradiction_marker_count",
#     ],
#     "semantic_coordination": [
#         "avg_pairwise_message_similarity",
#         "var_pairwise_message_similarity",
#         "num_messages",
#         "trace_word_length",
#         "avg_message_words",
#     ],
#     "fragmentation": [
#         "semantic_fragmentation_proxy",
#         "risk_conditioned_fragmentation",
#         "fragmentation_excess_over_benign",
#     ],
#     "all": TELEMETRY_FEATURES + [
#         "num_suspicious_tool_calls",
#         "used_any_suspicious_tool",
#         "semantic_fragmentation_proxy",
#         "risk_conditioned_fragmentation",
#         "fragmentation_excess_over_benign",
#     ],
# }

# def evaluate_feature_groups_loso(df, target_col="is_attack", model_kind="rf"):
#     rows = []

#     for group_name, features in FEATURE_GROUPS.items():
#         features = [c for c in features if c in df.columns]
#         if not features:
#             continue

#         res = leave_one_seed_out_eval(
#             df,
#             target_col=target_col,
#             features=features,
#             model_kind=model_kind,
#         )

#         if not res.empty:
#             rows.append({
#                 "feature_group": group_name,
#                 "model": model_kind,
#                 "target": target_col,
#                 "n_splits": len(res),
#                 "balanced_accuracy_mean": res["balanced_accuracy"].mean(),
#                 "balanced_accuracy_std": res["balanced_accuracy"].std(),
#                 "f1_mean": res["f1"].mean(),
#                 "f1_std": res["f1"].std(),
#                 "roc_auc_mean": res["roc_auc"].mean(),
#                 "roc_auc_std": res["roc_auc"].std(),
#             })

#     return pd.DataFrame(rows).sort_values("roc_auc_mean", ascending=False)

# ablation_loso_rf = evaluate_feature_groups_loso(
#     predictions_per_seed_df,
#     target_col="is_attack",
#     model_kind="rf",
# )

# display(ablation_loso_rf)